# Farmer Price Alert System (Mandi Insights Product)

FastAPI + Twilio + APScheduler + SQLite MVP.

## Install Dependencies

In [ ]:
!pip install fastapi uvicorn sqlalchemy twilio apscheduler python-dotenv requests

## database.py

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, declarative_base

DATABASE_URL = "sqlite:///farmers.db"

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

## models.py

In [ ]:
from sqlalchemy import Column, Integer, String, Float
from database import Base

class Farmer(Base):
    __tablename__ = "farmers"

    id = Column(Integer, primary_key=True)
    name = Column(String)
    phone = Column(String)
    crop = Column(String)
    mandi = Column(String)
    target_price = Column(Float)

## mandi_api.py

In [ ]:
import random

def get_mandi_price(crop, mandi):
    return round(random.uniform(1800, 3200), 2)

## scheduler.py

In [ ]:
from dotenv import load_dotenv
import os
from twilio.rest import Client

load_dotenv()

client = Client(
    os.getenv("TWILIO_SID"),
    os.getenv("TWILIO_TOKEN")
)

def send_sms(phone, message):
    client.messages.create(
        body=message,
        from_=os.getenv("TWILIO_PHONE"),
        to=phone
    )

## main.py

In [ ]:
from fastapi import FastAPI
from apscheduler.schedulers.background import BackgroundScheduler
from database import Base, engine, SessionLocal
from models import Farmer
from mandi_api import get_mandi_price
from scheduler import send_sms

app = FastAPI()

Base.metadata.create_all(bind=engine)

@app.post("/register")
def register(name: str, phone: str, crop: str, mandi: str, target_price: float):
    db = SessionLocal()
    farmer = Farmer(
        name=name,
        phone=phone,
        crop=crop,
        mandi=mandi,
        target_price=target_price
    )
    db.add(farmer)
    db.commit()
    return {"status": "Registered"}

@app.get("/farmers")
def farmers():
    db = SessionLocal()
    return db.query(Farmer).all()

def check_prices():
    db = SessionLocal()
    farmers = db.query(Farmer).all()
    for f in farmers:
        current = get_mandi_price(f.crop, f.mandi)
        if current >= f.target_price:
            msg = f"{f.crop} price in {f.mandi} is ₹{current}/quintal. Good time to sell."
            send_sms(f.phone, msg)

scheduler = BackgroundScheduler()
scheduler.add_job(check_prices, "interval", hours=1)
scheduler.start()

## Run the API

In [ ]:
!uvicorn main:app --reload

## Example API Request

In [ ]:
# POST /register
# name=Suman
# phone=+91XXXXXXXXXX
# crop=Wheat
# mandi=Nagpur
# target_price=2500